# Step By Step 07 - 风险因子模型
-------------------

> 在本篇中我们使用数据供应商提供的因子模型，进行alpha建模

## 一、数据展示

In [ ]:
import datetime as dt


### 1.1 因子暴露

这里我们展示某一天的因子截面值，这里我们只使用**风格因子**以及**行业因子**：

In [ ]:
import numpy as np


In [ ]:
import pyarrow as pa


In [ ]:
from alphamind.analysis.perfanalysis import perf_attribution_by_pos


### 1.2 因子风险模型

类似的，我们也可以获取因子的风险矩阵：

In [ ]:
from alphamind.portfolio.meanvariancebuilder import mean_variance_builder


In [ ]:
from alphamind.portfolio.riskmodel import FactorRiskModel


#### 1.2.1 因子协方差

In [ ]:
codes = ["A", "B", "C"]


#### 1.2.2 因子载荷

这个值实际上就是之前1.1的**因子暴露**

In [ ]:
covariance = pa.table({
    "Factor": ["BETA", "SIZE"],
    "BETA": [0.02, 0.003],
    "SIZE": [0.003, 0.01],
})


#### 1.2.3 因子特质风险

In [ ]:
exposure = pa.table({
    "code": codes,
    "BETA": [1.0, 0.8, 1.2],
    "SIZE": [0.2, 0.4, -0.1],
})


## 二、结合因子模型的alpha信号构造

首先我们给予 **Finance-Python** 提供的功能来构造一个目标因子：

In [ ]:
specific = pa.table({"code": codes, "idsync": [0.01, 0.015, 0.012]})


我们可以使用因子暴露对原始因子进行 **风格剥离**（中性化）,这里我们只选用 **市值以及Beta** 风格因子进行中性化：

In [ ]:
model = FactorRiskModel(covariance, exposure, specific)
profile = model.get_risk_profile(codes)


我们可以验证这两个因子确实被从因子中剥离了

In [ ]:
benchmark = np.full(3, 1.0 / 3.0)


## 三、组合构建

在本节中，我们使用上一节计算算得到的因子作为我们的单期预测，进行组合构建。

在这里我们使用经典的**单期均值方差模型**，但是会引入投资交易中经常需要考虑的各种限制：

1. 杠杆率的要求（不得过低或者过高）；
2. 单票集中度的要求（单票不可持仓过大）；
3. 行业板块集中度的要求（不可过高）；
4. 风格偏移的限制（对某些风格的偏好不可过高或过低）；
5. 在以上的限制下，最大化风险调整后收益。

在这里我们将展示一个例子，涵盖以上各个点；

In [ ]:
status, objective, weights = mean_variance_builder(
    np.array([0.02, 0.03, 0.01]),
    profile,
    benchmark,
    0.0,
    1.0,
    np.ones((3, 1)),
    (np.array([1.0]), np.array([1.0])),
)


我们将做控制以下控制：

1. 所以个股相对于基准，偏离不超过2%；
2. 所有行业板块，相对于基准偏离不超过3%；
3. 三个风格因子的风格偏离，不高于50%；
4. 杠杆率不低于80%，同时不高于120%

这里我们选用的基准为沪深300指数权重（做归一化）; 用户也可以直接使用0作为基准，这样的化，就是一个long-short组合。

下面进入我们整个环节中最复杂的一步，构建风险控制矩阵以及限制边界：

In [ ]:
# 个股限制
assert status in ("optimal", "optimal_inaccurate")


In [ ]:
# 我们这里使用一个带限制条件的优化器来解决这个组合优化问题：
trade_date = dt.datetime(2026, 1, 5)


让我们逐个检验限制条件是否被满足：

1. 个股限制

In [ ]:
active = pa.table({"trade_date": [trade_date] * 3, "code": codes, "weight": weights - benchmark})


2. 行业偏离

In [ ]:
returns = pa.table({"trade_date": [trade_date] * 3, "code": codes, "return": [0.01, -0.02, 0.015]})


3. 风格偏移

In [ ]:
risks = pa.table({
    "trade_date": [trade_date] * 3,
    "code": codes,
    "BETA": exposure["BETA"],
    "SIZE": exposure["SIZE"],
})


4. 杠杆率

In [ ]:
attribution = perf_attribution_by_pos(active, returns, risks)


5. 效用

> **注意**: 这里为了节省篇幅，我们简化了效用计算，实际的效用还包括风险的调整。但是简单的直接考虑期望的变化，也能看到优化器的作用。

In [ ]:
assert attribution.num_rows == 1


## 四、后记

这里我们非常细致的拆解了，使用风险因子模型，进行因子构造以及组合优化的过程。

实际上，alpha-mind提供了直接使用上面的算法进行回测的功能，用户没有必要自己手写上面每一步。

主要目的是通过这个例子，让用户了解alpha-mind因子建模的原理，鼓励用户自己基于alpha-mind的基础组件进行定制开发。本篇也无法涵盖alpha-mind在因子建模和组合优化上的所有功能。例如：对交易成本的控制等，留待以后的篇章向大家介绍。

In [ ]:
attribution
